In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 

os.makedirs('../outputs/images', exist_ok=True)

df = pd.read_csv('../data/preprocessed/final.csv')

In [2]:
df.head()

,Unnamed: 0,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,0,2,0.0,0.0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,0.0
1,1,0,1.0,1.0,4100000,12200000,8,417,2700000,2200000,8800000,3300000,1.0
2,2,3,0.0,0.0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,1.0
3,3,3,0.0,0.0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,1.0
4,4,5,1.0,1.0,9800000,24200000,20,382,12400000,8200000,29400000,5000000,1.0


In [3]:
df.drop(['Unnamed: 0'], axis=1, inplace=True)

In [4]:
df.head()

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,0.0,0.0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,0.0
1,0,1.0,1.0,4100000,12200000,8,417,2700000,2200000,8800000,3300000,1.0
2,3,0.0,0.0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,1.0
3,3,0.0,0.0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,1.0
4,5,1.0,1.0,9800000,24200000,20,382,12400000,8200000,29400000,5000000,1.0


In [9]:
df.shape

(4269, 12)

In [6]:
X = df.drop(columns=['loan_status'])
y = df['loan_status']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import warnings
warnings.filterwarnings('ignore')
import time

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder 
from sklearn.preprocessing import StandardScaler , MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error 
from sklearn.model_selection import StratifiedKFold, cross_validate

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import (
    BernoulliNB,
    CategoricalNB,
    ComplementNB,
    GaussianNB,
    MultinomialNB
)
from sklearn.tree import DecisionTreeClassifier , ExtraTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [15]:
# 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision']


clf_lr = LogisticRegression(max_iter=1000, random_state=42)
clf_rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
clf_lgbm = LGBMClassifier(n_estimators=50, random_state=42, verbose=-1, n_jobs=-1)

models = {
    # Linear & Distance-based
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
   
    "SVC (Linear approx)": SVC(probability=True, max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    
    # Naive Bayes
    "Gaussian NB": GaussianNB(),
    "Bernoulli NB": BernoulliNB(),
    # Multinomial/Complement require non-negative inputs
    "Multinomial NB": Pipeline([('minmax', MinMaxScaler()), ('nb', MultinomialNB())]),
    "Complement NB": Pipeline([('minmax', MinMaxScaler()), ('nb', ComplementNB())]),
    
    # Trees
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Extra Tree Decision": ExtraTreeClassifier(random_state=42),
    
    # Standard Ensembles
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Extra Trees ": ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Bagging (Tree)": BaggingClassifier(random_state=42, n_jobs=-1),
    
    # High-Performance Boosters
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbose=-1, n_jobs=-1),
    "CatBoost": CatBoostClassifier(iterations=100, verbose=0, random_state=42),
    
    # Meta Ensembles
    "Voting Classifier": VotingClassifier(
        estimators=[('lr', clf_lr), ('rf', clf_rf), ('lgb', clf_lgbm)],
        voting='soft'
    ),
    "Stacking Classifier": StackingClassifier(
        estimators=[('rf', clf_rf), ('lgb', clf_lgbm)],
        final_estimator=LogisticRegression(),
        cv=3,
        n_jobs=-1
    )
}


results = []
all_pipelines = {}


for name, model in models.items():
    print(f"🔄 Running 5-Fold CV for {name}...")
    start_time = time.time()
    
    pipe = Pipeline([
        ('model', model)
    ])
    
    try:
        cv_scores = cross_validate(
            pipe,
            X,
            y,
            cv=skf,
            scoring=scoring_metrics,
            n_jobs=-1,
            error_score='raise'
        )
        elapsed_time = time.time() - start_time
        all_pipelines[name] = pipe

        results.append({
            'Algorithm': name,
            'Mean_Accuracy': np.round(np.mean(cv_scores['test_accuracy']), 4),
            'Std_Accuracy': np.round(np.std(cv_scores['test_accuracy']), 4),
            'Mean_Precision': np.round(np.mean(cv_scores['test_precision']), 4),
            'Std_Precision': np.round(np.std(cv_scores['test_precision']), 4),
            'CV_Time_s': round(elapsed_time, 2)
        })
        print(f"✓ Completed {name} in {round(elapsed_time, 2)}s")
    except Exception as e:
        print(f"✗ Failed {name}: {str(e)[:60]}...")


performance_df = pd.DataFrame(results).sort_values(
    by=['Mean_Precision', 'Mean_Accuracy'], ascending=False
).reset_index(drop=True)

print("\n" + "="*50)
print("=== Model Performance across 5-Folds ===")
print("="*50)
print(performance_df.to_string())

if not performance_df.empty:
    best_model = performance_df.iloc[0]
    print("\n" + "-"*50)
    print("Best Model:           ", best_model['Algorithm'])
    print("Best Mean Accuracy:   ", best_model['Mean_Accuracy'])
    print("Best Mean Precision:  ", best_model['Mean_Precision'])
    print("-"*50)

🔄 Running 5-Fold CV for Logistic Regression...
✓ Completed Logistic Regression in 3.71s
🔄 Running 5-Fold CV for SVC (Linear approx)...
✓ Completed SVC (Linear approx) in 4.98s
🔄 Running 5-Fold CV for KNN...
✓ Completed KNN in 2.36s
🔄 Running 5-Fold CV for Gaussian NB...
✓ Completed Gaussian NB in 0.05s
🔄 Running 5-Fold CV for Bernoulli NB...
✓ Completed Bernoulli NB in 0.04s
🔄 Running 5-Fold CV for Multinomial NB...
✓ Completed Multinomial NB in 0.05s
🔄 Running 5-Fold CV for Complement NB...
✓ Completed Complement NB in 0.05s
🔄 Running 5-Fold CV for Decision Tree...
✓ Completed Decision Tree in 0.12s
🔄 Running 5-Fold CV for Extra Tree Decision...
✓ Completed Extra Tree Decision in 0.1s
🔄 Running 5-Fold CV for Random Forest...
✓ Completed Random Forest in 0.58s
🔄 Running 5-Fold CV for Extra Trees ...
✓ Completed Extra Trees  in 0.51s
🔄 Running 5-Fold CV for AdaBoost...
✓ Completed AdaBoost in 0.34s
🔄 Running 5-Fold CV for Gradient Boosting...
✓ Completed Gradient Boosting in 0.86s
🔄 Run

In [18]:
performance_df.sort_values(by=['Mean_Precision', 'Mean_Accuracy'], ascending=False)

,Algorithm,Mean_Accuracy,Std_Accuracy,Mean_Precision,Std_Precision,CV_Time_s
0,Voting Classifier,0.9843,0.0045,0.9950,0.0051,5.78
1,LightGBM,0.9864,0.0035,0.9875,0.0067,2.51
2,Stacking Classifier,0.9862,0.0026,0.9875,0.0073,6.08
3,Bagging (Tree),0.9831,0.0038,0.9874,0.0033,0.18
4,Random Forest,0.9824,0.0051,0.9856,0.0074,0.58
5,XGBoost,0.9852,0.0030,0.9850,0.0060,0.23
6,Gradient Boosting,0.9794,0.0037,0.9812,0.0064,0.86
7,Decision Tree,0.9775,0.0030,0.9662,0.0037,0.12
8,Extra Trees,0.9679,0.0066,0.9624,0.0056,0.51
9,AdaBoost,0.9667,0.0046,0.9608,0.0126,0.34
